# Nick-Default Generalizability Data Sweep Quickcheck

This notebook inspects the `nf_generalize_nick_data` sweep:

- u128 Nick-default training recipe
- combined `LH + CV`
- combined `z=0.0, 1.0, 2.0`
- `zthin=8`, so each 3D cube contributes 16 2D slices
- dataset sizes from `2^6` to `2^15` 2D slices

It first audits checkpoints and source allocation. Once samples exist, it computes one-point statistics, P(k), and image panels. The final PCA nearest-neighbor memorization/generalization diagnostic is loaded from the offline full-reference script output.


## Run Commands

Training is already done if all jobs in `50387266` completed. Next generate raw train-full samples:

```bash
cd /home/jiamingp/diffusion_models_repo
python scripts/prepare_nf_generalize_nick_data_configs.py --project-dir "$PWD" --check-only
sbatch -A huterer0 scripts/slurm/sample_nf_generalize_nick_data_array.sbatch
```

The default sample job writes 512 generated maps per run:

`results/nf_generalize_nick_data/samples/{run_name}_seed123_raw_train_full.npz`

For a fast smoke sample only:

```bash
NUM_SAMPLES=64 sbatch -A huterer0 --array=0,5,9 scripts/slurm/sample_nf_generalize_nick_data_array.sbatch
```


After samples finish, run the full-reference PCA nearest-neighbor diagnostic offline:

```bash
sbatch -A huterer0 scripts/slurm/analyze_nf_generalize_pca_full_nn.sbatch
```

This writes:

`results/nf_generalize_nick_data/tables/nf_generalize_nick_data_pca_full_nn_metrics.csv`

and the paper-style PCA GL figure:

`results/nf_generalize_nick_data/quickcheck/nf_generalize_nick_data_pca_full_nn_paper_style_gl_curves.png`


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'simdiff_eval').exists()), PROJECT_CANDIDATES[0])
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.io import as_nchw, load_real_from_config
from simdiff_eval.metrics import batch_power_spectra, field_histogram, power_spectrum_summary

SWEEP_NAME = 'nf_generalize_nick_data'
MANIFEST_PATH = PROJECT_DIR / 'local' / SWEEP_NAME / 'manifest.json'
CONFIG_DIR = PROJECT_DIR / 'local' / SWEEP_NAME / 'configs'
CHECKPOINT_ROOT = Path(os.environ.get('NF_GEN_NICK_CHECKPOINT_ROOT', f'/scratch/huterer_root/huterer0/jiamingp/saved_runs/{SWEEP_NAME}'))
SAMPLE_ROOT = PROJECT_DIR / 'results' / SWEEP_NAME / 'samples'
OUTPUT_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'quickcheck'
TABLE_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'tables'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(os.environ.get('NF_GEN_NICK_SEED', 123))
SAMPLE_LABEL = os.environ.get('NF_GEN_NICK_SAMPLE_LABEL', 'raw_train_full')
MAX_REAL_HIST = int(os.environ.get('NF_GEN_NICK_MAX_REAL_HIST', 1024))
MAX_REAL_PK = int(os.environ.get('NF_GEN_NICK_MAX_REAL_PK', 512))
MAX_GENERATED = int(os.environ.get('NF_GEN_NICK_MAX_GENERATED', 512))
PK_NBINS = int(os.environ.get('NF_GEN_NICK_PK_NBINS', 30))

PCA_N_COMPONENTS = int(os.environ.get('PCA_N_COMPONENTS', 32))
PCA_MAX_FIT_REAL = int(os.environ.get('PCA_MAX_FIT_REAL', 1024))
PCA_MAX_REAL_COMPARE = int(os.environ.get('PCA_MAX_REAL_COMPARE', 2048))
PCA_MAX_GENERATED = int(os.environ.get('PCA_MAX_GENERATED', 512))
PCA_REF_FRACTION = float(os.environ.get('PCA_REF_FRACTION', 0.75))
PCA_COPY_QUANTILES = [float(x) for x in os.environ.get('PCA_COPY_QUANTILES', '0.90,0.95,0.99').split(',')]
SIMILARITY_BATCH_SIZE = int(os.environ.get('SIMILARITY_BATCH_SIZE', 1024))
RUN_NOTEBOOK_CAPPED_PCA = os.environ.get('NF_GEN_NICK_RUN_CAPPED_PCA', '0') == '1'
PCA_FULL_NN_METRICS_PATH = TABLE_DIR / 'nf_generalize_nick_data_pca_full_nn_metrics.csv'
PCA_FULL_NN_SIMILARITY_FIG = OUTPUT_DIR / 'nf_generalize_nick_data_pca_full_nn_similarity_curves.png'
PCA_FULL_NN_COPY_FIG = OUTPUT_DIR / 'nf_generalize_nick_data_pca_full_nn_copy_fraction_curves.png'
PCA_FULL_NN_GL_FIG = OUTPUT_DIR / 'nf_generalize_nick_data_pca_full_nn_paper_style_gl_curves.png'

print('PROJECT_DIR =', PROJECT_DIR)
print('MANIFEST_PATH =', MANIFEST_PATH)
print('CHECKPOINT_ROOT =', CHECKPOINT_ROOT)
print('SAMPLE_ROOT =', SAMPLE_ROOT)


## Manifest, Data Allocation, Checkpoints, Samples

`dataset_size` is the number of 2D maps. `n_train_simulations` is the number of 3D cubes; with `zthin=8`, each cube gives 16 maps.


In [ ]:
def dataset_size(row: dict[str, Any]) -> int:
    return int(row.get('dataset_size', row.get('actual_2d', row.get('target_2d'))))


def config_path_for(row: dict[str, Any]) -> Path:
    raw = row.get('config') or f'local/{SWEEP_NAME}/configs/{row["run_name"]}.yaml'
    path = Path(raw)
    return path if path.is_absolute() else PROJECT_DIR / path


def sample_path_for(row: dict[str, Any]) -> Path:
    return SAMPLE_ROOT / f"{row['run_name']}_seed{SEED}_{SAMPLE_LABEL}.npz"


def latest_checkpoint_epoch(run_name: str) -> int | None:
    root = CHECKPOINT_ROOT / f'{run_name}_checkpoints'
    epochs = []
    for path in root.glob('checkpoint-epoch-*'):
        match = re.search(r'checkpoint-epoch-(\d+)$', path.name)
        if match:
            epochs.append(int(match.group(1)))
    return max(epochs) if epochs else None


def load_npz_array(path: Path) -> np.ndarray:
    z = np.load(path, mmap_mode='r')
    try:
        if 'samples' in z:
            arr = np.asarray(z['samples'])
        elif 'arr_0' in z:
            arr = np.asarray(z['arr_0'])
        else:
            arr = np.asarray(z[z.files[0]])
    finally:
        z.close()
    return as_nchw(arr)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f'Missing {MANIFEST_PATH}. Run scripts/prepare_nf_generalize_nick_data_configs.py first.')

rows = sorted(json.loads(MANIFEST_PATH.read_text()), key=dataset_size)
run_rows = []
source_rows = []
for idx, row in enumerate(rows):
    sample_path = sample_path_for(row)
    n_available = 0
    if sample_path.exists():
        try:
            n_available = len(load_npz_array(sample_path))
        except Exception as exc:
            print('Could not inspect sample:', sample_path, repr(exc))
    latest = latest_checkpoint_epoch(row['run_name'])
    run_rows.append({
        'idx': idx,
        'run_name': row['run_name'],
        'dataset_tag': row.get('dataset_tag'),
        'dataset_size': dataset_size(row),
        'n_train_simulations': row.get('n_train_simulations'),
        'zthin': row.get('zthin'),
        'slices_per_sim': row.get('slices_per_sim'),
        'latest_epoch': latest,
        'final_checkpoint': latest == 99,
        'sample_exists': sample_path.exists(),
        'n_available': n_available,
        'sample_path': str(sample_path),
    })
    for source in row.get('source_counts', []):
        source_rows.append({
            'idx': idx,
            'run_name': row['run_name'],
            'dataset_size': dataset_size(row),
            'source': source['tag'],
            'n_3d_cubes': int(source['n_samples']),
            'n_2d_slices': int(source['n_2d_slices']),
            'path': source.get('path'),
        })

run_df = pd.DataFrame(run_rows)
source_df = pd.DataFrame(source_rows)
display(run_df)
print('final checkpoints:', int(run_df['final_checkpoint'].sum()), '/', len(run_df))
print('sample files:', int(run_df['sample_exists'].sum()), '/', len(run_df))

source_pivot = source_df.pivot_table(index=['idx', 'run_name', 'dataset_size'], columns='source', values='n_3d_cubes', aggfunc='sum', fill_value=0)
display(source_pivot)


## Training Loss Curves

This is only an optimization-health check. It should not be used as the main sample-quality metric.


In [ ]:
def metric_candidates(run_name: str) -> list[Path]:
    root = CHECKPOINT_ROOT / f'{run_name}_checkpoints'
    paths = []
    paths.extend(sorted(root.glob('metrics_epoch_*.json')))
    paths.extend(sorted(root.glob('metrics.json')))
    for ckpt in sorted(root.glob('checkpoint-epoch-*')):
        paths.extend(sorted(ckpt.glob('metrics*.json')))
    return paths

metrics_by_run = {}
metric_rows = []
for row in rows:
    paths = metric_candidates(row['run_name'])
    metrics = None
    if paths:
        with paths[-1].open() as f:
            metrics = json.load(f)
        metrics_by_run[row['run_name']] = metrics
    epoch_loss = np.asarray((metrics or {}).get('epoch_loss', []), dtype=float)
    batch_loss = np.asarray((metrics or {}).get('batch_loss', []), dtype=float)
    metric_rows.append({
        'run_name': row['run_name'],
        'dataset_size': dataset_size(row),
        'metrics_path': str(paths[-1]) if paths else None,
        'n_epoch_loss': len(epoch_loss),
        'final_epoch_loss': float(epoch_loss[-1]) if len(epoch_loss) else np.nan,
        'n_batch_loss': len(batch_loss),
        'final_batch_loss': float(batch_loss[-1]) if len(batch_loss) else np.nan,
    })

metric_df = pd.DataFrame(metric_rows).sort_values('dataset_size')
display(metric_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for row in rows:
    metrics = metrics_by_run.get(row['run_name'])
    if not metrics:
        continue
    label = row.get('dataset_tag', row['run_name'])
    epoch_loss = np.asarray(metrics.get('epoch_loss', []), dtype=float)
    if len(epoch_loss):
        axes[0].plot(np.arange(len(epoch_loss)), epoch_loss, marker='o', ms=2.5, lw=1.3, label=label)
    batch_loss = np.asarray(metrics.get('batch_loss', []), dtype=float)
    if len(batch_loss):
        axes[1].plot(np.arange(len(batch_loss)), batch_loss, lw=0.8, alpha=0.8, label=label)
axes[0].set_title('epoch loss')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('mean MSE loss')
axes[1].set_title('batch loss')
axes[1].set_xlabel('optimizer step')
axes[1].set_ylabel('MSE loss')
for ax in axes:
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=8, ncol=2)
fig.tight_layout()
out = OUTPUT_DIR / 'nf_generalize_nick_data_training_loss.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
print('wrote', out)
plt.show()


## Load Samples

This section is empty until `sample_nf_generalize_nick_data_array.sbatch` finishes.


In [ ]:
def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= limit:
        return arr.copy()
    idx = np.linspace(0, len(arr) - 1, limit, dtype=int)
    return arr[idx].copy()

# Do not load the full real training arrays here. The largest run has 32768
# 2D maps, and keeping all rows in memory can kill the notebook kernel.
REAL_SLICE_CAP = int(os.environ.get(
    'NF_GEN_NICK_REAL_SLICE_CAP',
    max(MAX_REAL_HIST, MAX_REAL_PK, PCA_MAX_REAL_COMPARE, PCA_MAX_FIT_REAL),
))


def raw_sim_cap_for(row: dict[str, Any], slice_cap: int | None) -> int | None:
    if slice_cap is None:
        return None
    slices_per_sim = int(row.get('slices_per_sim') or max(1, 128 // int(row.get('zthin', 8))))
    cap = int(math.ceil(int(slice_cap) / slices_per_sim))
    total = row.get('n_train_simulations') or row.get('n_samples_simulations')
    if total is not None:
        cap = min(cap, int(total))
    return max(1, cap)


real_cache: dict[tuple[str, int], np.ndarray] = {}


def load_real_for_row(row: dict[str, Any], slice_cap: int) -> np.ndarray:
    key = (row['run_name'], int(slice_cap))
    if key not in real_cache:
        config_path = config_path_for(row)
        raw_cap = raw_sim_cap_for(row, slice_cap)
        real = as_nchw(load_real_from_config(config_path, max_raw_samples=raw_cap))
        real_cache[key] = evenly_limit(real, slice_cap)
    return real_cache[key]


loaded: dict[str, dict[str, Any]] = {}

for row in rows:
    sample_path = sample_path_for(row)
    if not sample_path.exists():
        continue
    generated = evenly_limit(load_npz_array(sample_path), MAX_GENERATED)
    real = load_real_for_row(row, REAL_SLICE_CAP)
    loaded[row['run_name']] = {
        'spec': row,
        'real': real,
        'generated': generated,
        'sample_path': sample_path,
    }

loaded_rows = []
for run_name, bundle in loaded.items():
    row = bundle['spec']
    loaded_rows.append({
        'run_name': run_name,
        'dataset_tag': row.get('dataset_tag'),
        'dataset_size': dataset_size(row),
        'n_real_loaded': len(bundle['real']),
        'real_slice_cap': REAL_SLICE_CAP,
        'raw_sim_cap': raw_sim_cap_for(row, REAL_SLICE_CAP),
        'n_generated': len(bundle['generated']),
        'sample_path': str(bundle['sample_path']),
    })
loaded_df = pd.DataFrame(loaded_rows).sort_values('dataset_size') if loaded_rows else pd.DataFrame()
display(loaded_df)
print('loaded sample rows:', len(loaded))
if not loaded:
    print('No samples found yet. Submit: sbatch -A huterer0 scripts/slurm/sample_nf_generalize_nick_data_array.sbatch')


## One-Point Statistics And P(k)

Lower `hist_l1` and `pk_log10_mae` are better. Ratios near 1 are better.


In [ ]:
def onepoint_metrics(real: np.ndarray, generated: np.ndarray, bins: int = 120) -> dict[str, float]:
    real_l = evenly_limit(real, MAX_REAL_HIST)
    gen_l = evenly_limit(generated, MAX_GENERATED)
    rh = field_histogram(real_l, bins=bins)
    gh = field_histogram(gen_l, bins=bins)
    edges = np.asarray(rh['bin_edges'])
    width = float(np.mean(np.diff(edges)))
    hist_l1 = float(np.sum(np.abs(np.asarray(rh['hist']) - np.asarray(gh['hist']))) * width)
    return {
        'real_mean': rh['mean'],
        'generated_mean': gh['mean'],
        'real_std': rh['std'],
        'generated_std': gh['std'],
        'std_ratio': gh['std'] / max(rh['std'], 1e-30),
        'real_q01': rh['q01'],
        'generated_q01': gh['q01'],
        'real_q99': rh['q99'],
        'generated_q99': gh['q99'],
        'hist_l1': hist_l1,
    }

metric_rows = []
for run_name, bundle in loaded.items():
    row = bundle['spec']
    real = evenly_limit(bundle['real'], MAX_REAL_PK)
    generated = evenly_limit(bundle['generated'], MAX_GENERATED)
    metric_rows.append({
        'run_name': run_name,
        'dataset_tag': row.get('dataset_tag'),
        'dataset_size': dataset_size(row),
        'n_generated': len(generated),
        **onepoint_metrics(bundle['real'], bundle['generated']),
        **power_spectrum_summary(real, generated, nbins=PK_NBINS),
    })

metrics_df = pd.DataFrame(metric_rows).sort_values('dataset_size') if metric_rows else pd.DataFrame()
if len(metrics_df):
    display(metrics_df)
    out = TABLE_DIR / 'nf_generalize_nick_data_raw_train_full_metrics.csv'
    metrics_df.to_csv(out, index=False)
    print('wrote', out)
else:
    print('No metrics yet because no samples are loaded.')


In [ ]:
if len(metrics_df):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    x = metrics_df['dataset_size']
    axes[0].plot(x, metrics_df['hist_l1'], marker='o')
    axes[0].set_ylabel('one-point histogram L1')
    axes[1].plot(x, metrics_df['pk_log10_mae'], marker='o')
    axes[1].set_ylabel('P(k) log10 MAE')
    axes[2].plot(x, metrics_df['std_ratio'], marker='o')
    axes[2].axhline(1.0, color='black', ls=':', lw=1.2)
    axes[2].set_ylabel('generated / real std')
    for ax in axes:
        ax.set_xscale('log', base=2)
        ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: rf'$2^{{{int(round(np.log2(v)))}}}$' if v > 0 and np.isclose(v, 2 ** round(np.log2(v))) else f'{v:g}'))
        ax.set_xlabel('training set size N')
        ax.grid(alpha=0.25)
    fig.suptitle('raw train_full physics metrics versus dataset size')
    fig.tight_layout()
    out = OUTPUT_DIR / 'nf_generalize_nick_data_metric_curves.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('wrote', out)
    plt.show()


## One-Point And P(k) Panels


In [ ]:
if loaded:
    items = sorted(loaded.items(), key=lambda kv: dataset_size(kv[1]['spec']))
    ncols = min(5, len(items))
    nrows = math.ceil(len(items) / ncols)
    fig, axes = plt.subplots(2 * nrows, ncols, figsize=(4.1 * ncols, 5.6 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')

    for col, (run_name, bundle) in enumerate(items):
        block = col // ncols
        pos = col % ncols
        row = bundle['spec']
        real = evenly_limit(bundle['real'], MAX_REAL_PK)
        generated = evenly_limit(bundle['generated'], MAX_GENERATED)

        ax = axes[2 * block, pos]
        ax.axis('on')
        rh = field_histogram(evenly_limit(real, MAX_REAL_HIST))
        gh = field_histogram(evenly_limit(generated, MAX_GENERATED))
        edges = np.asarray(rh['bin_edges'])
        centers = 0.5 * (edges[:-1] + edges[1:])
        ax.plot(centers, rh['hist'], color='black', label='real')
        ax.plot(centers, gh['hist'], color='tab:blue', label='generated')
        ax.set_yscale('log')
        ax.set_title(f"{row.get('dataset_tag')} N={dataset_size(row)}")
        ax.set_xlabel('normalized field value')
        ax.set_ylabel('density')
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)

        ax = axes[2 * block + 1, pos]
        ax.axis('on')
        pk_real, kbins = batch_power_spectra(real, nbins=PK_NBINS)
        pk_gen, _ = batch_power_spectra(generated, nbins=PK_NBINS)
        ratio_pct = 100 * (np.nanmean(pk_gen, axis=0) - np.nanmean(pk_real, axis=0)) / np.clip(np.nanmean(pk_real, axis=0), 1e-30, None)
        ax.plot(kbins, ratio_pct, marker='o', color='tab:blue')
        ax.axhline(0, color='black', ls=':', lw=1.2)
        ax.set_xlabel('k bin')
        ax.set_ylabel('P(k) error [%]')
        ax.grid(alpha=0.2)

    fig.suptitle('one-point and P(k) mismatch by dataset size')
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    out = OUTPUT_DIR / 'nf_generalize_nick_data_physics_panels.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('wrote', out)
    plt.show()
else:
    print('No samples loaded yet.')


## Real Versus Generated Images


In [ ]:
if loaded:
    items = sorted(loaded.items(), key=lambda kv: dataset_size(kv[1]['spec']))
    ncols = min(5, len(items))
    nrows = math.ceil(len(items) / ncols)
    fig, axes = plt.subplots(2 * nrows, ncols, figsize=(3.2 * ncols, 6.0 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    vals = np.concatenate([np.asarray(v['real'][:1]).ravel() for _, v in items] + [np.asarray(v['generated'][:1]).ravel() for _, v in items])
    vmin, vmax = np.nanpercentile(vals, [1, 99])
    for col, (run_name, bundle) in enumerate(items):
        block = col // ncols
        pos = col % ncols
        row = bundle['spec']
        axes[2 * block, pos].imshow(bundle['real'][0, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        axes[2 * block, pos].set_title(f"{row.get('dataset_tag')} real")
        axes[2 * block + 1, pos].imshow(bundle['generated'][0, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        axes[2 * block + 1, pos].set_title('generated')
    fig.suptitle('real vs generated examples')
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    out = OUTPUT_DIR / 'nf_generalize_nick_data_images.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('wrote', out)
    plt.show()
else:
    print('No samples loaded yet.')


## PCA Nearest-Neighbor Generalizability Diagnostic

The final PCA diagnostic is now computed offline with the full training reference set for each dataset size, using Nick's requested metric

`s_j = max_i cos(z_j, z_train[i])`.

The notebook quick-check PCA below is capped for memory and is optional only. Use the offline table and figures in this section for the clean result.


### Full-Reference PCA NN Results

The offline script computes Nick's requested nearest-training statistic with the full training reference set for each dataset size:

```text
z(x) = L2-normalized PCA embedding of image x
s_gen[j] = max_i cos(z(generated_j), z(train_i))
s_val[j] = max_i cos(z(heldout_real_j), z(train_i))
s_ref[j] = max_{i != j} cos(z(train_j), z(train_i))
```

`heldout_real` means real CAMELS slices from the same source files that were **not** included in the training subset for that `N`. It is a baseline for how close genuine unseen real samples are to the training set.

The paper-style generalizability score is a thresholded version of `s_gen`:

```text
GL(tau) = 1 - mean_j[s_gen[j] > tau]
```

[Zhang et al. 2024, arXiv:2310.05264](https://arxiv.org/abs/2310.05264) use SSCD features and a fixed copy threshold, estimated with many generated samples. Here we are only trying a PCA approximation, so interpret it as a diagnostic rather than a faithful reproduction.


### What `GL(tau)` Means

`tau` is a chosen similarity cutoff. For each generated image, we compute its nearest-training similarity:

```text
s_gen[j] = max_i cos(z(generated_j), z(train_i))
```

Then:

```text
fraction_above_threshold = mean_j[s_gen[j] > tau]
GL(tau) = 1 - fraction_above_threshold
```

So `GL(tau)` is the fraction of generated samples that are **not** too close to the training set under that threshold.

- `GL(tau) = 1`: none of the generated samples are above the copy threshold.
- `GL(tau) = 0`: all generated samples are above the copy threshold.
- Lower `GL` means more copy-like under this feature/threshold.
- Higher `GL` does **not** automatically mean good generalization, because bad/off-manifold samples can also be far from training data.

This is why the PCA/SSCD nearest-neighbor plot is a copy audit, not a full sample-quality metric. We still need one-point statistics, P(k), images, and ideally a PFD/teacher-reference style diagnostic.


In [ ]:
from IPython.display import Image, display

pca_full_nn_df = pd.DataFrame()
if PCA_FULL_NN_METRICS_PATH.exists():
    pca_full_nn_df = pd.read_csv(PCA_FULL_NN_METRICS_PATH).sort_values('dataset_size')
    # Backward-compatible GL columns if the CSV was made before the script update.
    for suffix in ['q90', 'q95', 'q99']:
        gen_col = f'gen_copy_fraction_{suffix}'
        val_col = f'val_copy_fraction_{suffix}'
        if gen_col in pca_full_nn_df and f'gen_gl_{suffix}' not in pca_full_nn_df:
            pca_full_nn_df[f'gen_gl_{suffix}'] = 1.0 - pca_full_nn_df[gen_col]
        if val_col in pca_full_nn_df and f'val_gl_{suffix}' not in pca_full_nn_df:
            pca_full_nn_df[f'val_gl_{suffix}'] = 1.0 - pca_full_nn_df[val_col]

    display_cols = [
        'dataset_tag', 'dataset_size', 'n_train_ref', 'n_val_real', 'n_generated',
        'gen_nn_median', 'val_nn_median', 'gen_nn_q99', 'val_nn_q99', 'threshold_q99',
        'gen_copy_fraction_q99', 'gen_gl_q99', 'val_copy_fraction_q99', 'val_gl_q99',
    ]
    display_cols = [c for c in display_cols if c in pca_full_nn_df.columns]
    display(pca_full_nn_df[display_cols])
    print('loaded', PCA_FULL_NN_METRICS_PATH)
else:
    print('Missing full-reference PCA NN metrics:')
    print(PCA_FULL_NN_METRICS_PATH)
    print('Run: sbatch -A huterer0 scripts/slurm/analyze_nf_generalize_pca_full_nn.sbatch')

for fig_path in [PCA_FULL_NN_SIMILARITY_FIG, PCA_FULL_NN_COPY_FIG, PCA_FULL_NN_GL_FIG]:
    if fig_path.exists():
        print(fig_path)
        display(Image(filename=str(fig_path)))
    else:
        print('missing figure:', fig_path)


### Why This Does Not Automatically Reproduce Fig. 2

The PCA diagnostic can fail to show the expected memorization-to-generalization curve for three reasons:

1. **Different feature space.** The paper uses SSCD copy-detection features. This notebook uses PCA on normalized CAMELS maps. PCA captures dominant variance modes, but it is not trained to detect near-duplicate images.

2. **Different score.** The paper's GL is `1 - copy_fraction` under a fixed copy threshold. The q99 panel is not GL; it is the 99th percentile of nearest-neighbor similarity. It answers: "how close are the most training-like generated samples?"

3. **Bad samples can look non-memorized.** A low-quality/off-manifold generated field can be far from every training image, giving high apparent GL even though it is not a good generalizer. [Zhang et al. 2026, arXiv:2505.20123](https://arxiv.org/abs/2505.20123) explicitly points out this failure mode for dissimilarity-to-training metrics.

So the right interpretation is:

- increasing `gen_nn_median` with `N` means generated samples are becoming more real-like / closer to the CAMELS manifold;
- `gen_copy_fraction_q99 = 0` means no generated sample crosses the PCA near-copy threshold;
- this does **not** prove paper-style generalization, because PCA distance from training can confuse poor samples with novel samples.


In [ ]:
if len(pca_full_nn_df):
    def xfmt(v, _):
        if v <= 0:
            return ''
        exp = int(round(np.log2(v)))
        return rf'$2^{{{exp}}}$' if np.isclose(v, 2 ** exp) else f'{v:g}'

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), sharex=True)
    x = pca_full_nn_df['dataset_size'].astype(float)

    for suffix in ['q95', 'q99']:
        gen_col = f'gen_gl_{suffix}'
        val_col = f'val_gl_{suffix}'
        if gen_col in pca_full_nn_df:
            axes[0].plot(x, pca_full_nn_df[gen_col], marker='o', label=f'generated, train-real {suffix}')
        if val_col in pca_full_nn_df:
            axes[0].plot(x, pca_full_nn_df[val_col], marker='o', ls='--', alpha=0.75, label=f'held-out real, train-real {suffix}')
    axes[0].set_title('PCA GL with adaptive real-real threshold')
    axes[0].set_ylabel('GL = 1 - fraction above threshold')

    fixed_cols = sorted(
        [c for c in pca_full_nn_df.columns if c.startswith('gen_gl_fixed_')],
        key=lambda c: float(c.rsplit('_', 1)[-1].replace('p', '.')),
    )
    if fixed_cols:
        for col in fixed_cols:
            tau = col.rsplit('_', 1)[-1].replace('p', '.')
            axes[1].plot(x, pca_full_nn_df[col], marker='o', label=f'tau={tau}')
        axes[1].set_title('PCA GL with fixed similarity thresholds')
    else:
        axes[1].text(0.5, 0.5, 'fixed-threshold columns missing\nrerun analyze_nf_generalize_pca_full_nn.sbatch',
                     ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title('fixed-threshold GL not available yet')
    axes[1].set_ylabel('GL = 1 - fraction above fixed tau')

    for ax in axes:
        ax.set_xscale('log', base=2)
        ax.xaxis.set_major_formatter(ticker.FuncFormatter(xfmt))
        ax.set_ylim(-0.03, 1.03)
        ax.set_xlabel('training dataset size N')
        ax.grid(alpha=0.25)
        ax.legend(fontsize=9)

    fig.suptitle('Attempted paper-style generalizability plot using PCA features')
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    out = OUTPUT_DIR / 'nf_generalize_nick_data_pca_gl_attempt_from_csv.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('wrote', out)
    plt.show()


In [ ]:
def flatten_images(images: np.ndarray) -> np.ndarray:
    arr = as_nchw(images).astype(np.float32, copy=False)
    return arr.reshape(len(arr), -1)


class PCAEncoder:
    def __init__(self, mean: np.ndarray, scale: np.ndarray, components: np.ndarray, evr: np.ndarray):
        self.mean = mean.astype(np.float32)
        self.scale = scale.astype(np.float32)
        self.components = components.astype(np.float32)
        self.explained_variance_ratio = evr.astype(np.float32)

    def transform(self, images: np.ndarray) -> np.ndarray:
        x = flatten_images(images)
        x = (x - self.mean) / self.scale
        return x @ self.components.T


def fit_pca_encoder(real: np.ndarray, n_components: int = 32, max_fit: int = 1024) -> PCAEncoder:
    x = flatten_images(evenly_limit(real, max_fit))
    mean = x.mean(axis=0, keepdims=True)
    scale = x.std(axis=0, keepdims=True)
    scale = np.where(scale < 1e-6, 1.0, scale)
    x = ((x - mean) / scale).astype(np.float32, copy=False)
    n_components = int(min(n_components, x.shape[0] - 1, x.shape[1]))
    if n_components < 2:
        raise ValueError('Need at least 3 real samples to fit PCA.')
    try:
        from sklearn.decomposition import PCA
        pca = PCA(n_components=n_components, svd_solver='randomized', random_state=0)
        pca.fit(x)
        components = pca.components_
        evr = pca.explained_variance_ratio_
    except Exception as exc:
        print('sklearn PCA unavailable or failed; using numpy SVD:', repr(exc))
        _, s, vt = np.linalg.svd(x, full_matrices=False)
        components = vt[:n_components]
        var = (s ** 2) / max(len(x) - 1, 1)
        evr = var[:n_components] / np.clip(var.sum(), 1e-30, None)
    return PCAEncoder(mean.squeeze(0), scale.squeeze(0), components, evr)


def l2_normalize(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)


def nearest_self_similarity(z: np.ndarray, batch_size: int = 1024) -> np.ndarray:
    z = np.asarray(z, dtype=np.float32)
    vals = []
    n = len(z)
    for start in range(0, n, batch_size):
        stop = min(start + batch_size, n)
        sim = z[start:stop] @ z.T
        sim[np.arange(stop - start), np.arange(start, stop)] = -np.inf
        vals.append(np.max(sim, axis=1))
    return np.concatenate(vals)


def nearest_cross_similarity(query_z: np.ndarray, ref_z: np.ndarray, batch_size: int = 1024) -> np.ndarray:
    vals = []
    for start in range(0, len(query_z), batch_size):
        stop = min(start + batch_size, len(query_z))
        sim = query_z[start:stop] @ ref_z.T
        vals.append(np.max(sim, axis=1))
    return np.concatenate(vals)


def deterministic_real_split(real: np.ndarray, ref_fraction: float = 0.75) -> tuple[np.ndarray, np.ndarray]:
    real = as_nchw(real)
    n = len(real)
    if n < 4:
        raise ValueError(f'Need at least 4 real slices for ref/validation split, got {n}.')
    idx = np.arange(n)
    step = max(2, int(round(1 / max(1e-6, 1 - ref_fraction))))
    val_mask = (idx % step) == 0
    if val_mask.sum() < 2 or (~val_mask).sum() < 2:
        val_mask = idx >= int(round(n * ref_fraction))
    return real[~val_mask].copy(), real[val_mask].copy()


def qstats(values: np.ndarray, prefix: str) -> dict[str, float]:
    values = np.asarray(values)
    return {
        f'{prefix}_median': float(np.median(values)),
        f'{prefix}_q90': float(np.quantile(values, 0.90)),
        f'{prefix}_q95': float(np.quantile(values, 0.95)),
        f'{prefix}_q99': float(np.quantile(values, 0.99)),
    }


In [ ]:
pca_records = []
similarity_cache = {}
pca_df = pd.DataFrame()

if not RUN_NOTEBOOK_CAPPED_PCA:
    print('Skipping capped notebook PCA quick-check. Set NF_GEN_NICK_RUN_CAPPED_PCA=1 before launching Jupyter to run it.')
elif loaded:
    largest = max(loaded.values(), key=lambda b: dataset_size(b['spec']))
    print('Fitting PCA on largest available capped row:', largest['spec']['run_name'])
    fit_real = evenly_limit(largest['real'], PCA_MAX_FIT_REAL)
    encoder = fit_pca_encoder(fit_real, n_components=PCA_N_COMPONENTS, max_fit=PCA_MAX_FIT_REAL)
    print('PCA explained variance sum:', float(encoder.explained_variance_ratio.sum()))

    for run_name, bundle in sorted(loaded.items(), key=lambda kv: dataset_size(kv[1]['spec'])):
        row = bundle['spec']
        real_all = evenly_limit(bundle['real'], PCA_MAX_REAL_COMPARE)
        generated = evenly_limit(bundle['generated'], PCA_MAX_GENERATED)
        real_ref, real_val = deterministic_real_split(real_all, PCA_REF_FRACTION)
        ref_z = l2_normalize(encoder.transform(real_ref))
        val_z = l2_normalize(encoder.transform(real_val))
        gen_z = l2_normalize(encoder.transform(generated))

        ref_nn = nearest_self_similarity(ref_z, SIMILARITY_BATCH_SIZE)
        val_nn = nearest_cross_similarity(val_z, ref_z, SIMILARITY_BATCH_SIZE)
        gen_nn = nearest_cross_similarity(gen_z, ref_z, SIMILARITY_BATCH_SIZE)
        finite_ref = ref_nn[np.isfinite(ref_nn)]
        rec = {
            'run_name': run_name,
            'dataset_tag': row.get('dataset_tag'),
            'dataset_size': dataset_size(row),
            'n_generated': len(generated),
            'n_real_ref': len(real_ref),
            'n_real_val': len(real_val),
            'pca_components': len(encoder.explained_variance_ratio),
            'pca_explained_variance_sum': float(encoder.explained_variance_ratio.sum()),
            **qstats(finite_ref, 'ref_nn'),
            **qstats(val_nn, 'val_nn'),
            **qstats(gen_nn, 'gen_nn'),
        }
        for q in PCA_COPY_QUANTILES:
            key = int(round(q * 100))
            threshold = float(np.quantile(finite_ref, q))
            rec[f'threshold_q{key}'] = threshold
            rec[f'gen_copy_frac_q{key}'] = float(np.mean(gen_nn >= threshold))
            rec[f'val_copy_frac_q{key}'] = float(np.mean(val_nn >= threshold))
        pca_records.append(rec)
        similarity_cache[run_name] = {'ref_nn': ref_nn, 'val_nn': val_nn, 'gen_nn': gen_nn}
        print(run_name, 'gen_median=', rec['gen_nn_median'], 'ref_q99=', rec['ref_nn_q99'])

    pca_df = pd.DataFrame(pca_records).sort_values('dataset_size') if pca_records else pd.DataFrame()
    if len(pca_df):
        display(pca_df)
        out = TABLE_DIR / 'nf_generalize_nick_data_pca_nn_metrics_capped_notebook.csv'
        pca_df.to_csv(out, index=False)
        print('wrote', out)
else:
    print('No capped notebook PCA diagnostics because no samples are loaded.')


In [ ]:
if len(pca_df):
    def xfmt(v, _):
        if v <= 0:
            return ''
        exp = int(round(np.log2(v)))
        return rf'$2^{{{exp}}}$' if np.isclose(v, 2 ** exp) else f'{v:g}'

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    x = pca_df['dataset_size']
    axes[0].plot(x, pca_df['gen_nn_median'], marker='o', label='generated median')
    axes[0].plot(x, pca_df['val_nn_median'], marker='x', ls='--', label='real validation median')
    axes[0].set_ylabel('PCA nearest-neighbor cosine')
    axes[0].set_title('median nearest-training similarity')

    key = 99 if 'threshold_q99' in pca_df.columns else int(round(PCA_COPY_QUANTILES[-1] * 100))
    axes[1].plot(x, pca_df[f'gen_nn_q{key}'], marker='o', label=f'generated q{key}')
    axes[1].plot(x, pca_df[f'threshold_q{key}'], marker='s', ls=':', label=f'real-real q{key} threshold')
    axes[1].set_ylabel('PCA nearest-neighbor cosine')
    axes[1].set_title(f'generated q{key} versus real-real q{key}')

    for ax in axes:
        ax.set_xscale('log', base=2)
        ax.xaxis.set_major_formatter(ticker.FuncFormatter(xfmt))
        ax.set_xlabel('training set size N')
        ax.grid(alpha=0.25)
        ax.legend(fontsize=9)
    fig.suptitle('PCA nearest-neighbor generalizability diagnostic')
    fig.tight_layout()
    out = OUTPUT_DIR / 'nf_generalize_nick_data_pca_similarity_curves.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('wrote', out)
    plt.show()


## Interpretation

- Use `nf_generalize_nick_data_pca_full_nn_metrics.csv` for the clean PCA nearest-neighbor result. It uses the full training reference set for each `N` and compares generated samples against held-out real validation samples.
- `gen_nn_median` increasing with `N` means generated samples are moving closer to the real-data manifold as sample quality improves. It is not by itself evidence of memorization.
- Near-copy behavior should show up as generated samples crossing the real-real leave-one-out thresholds. In the full-reference run, `gen_copy_fraction_q99 = 0` means no generated sample crosses the q99 near-copy threshold under this PCA embedding.
- The closest paper-style score in this notebook is `GL(tau) = 1 - mean[max_i similarity(gen_j, train_i) > tau]`. The paper uses SSCD and a fixed threshold; this notebook currently uses PCA, so it is only an approximation.
- If PCA GL stays high at low `N`, do **not** read that as successful generalization. Low-quality/off-manifold samples can be far from training images and therefore look non-copied.
- Sample quality still needs the one-point statistics, P(k), and image panels above.
